# Cocoa Price Shocks and Macroeconomic Adjustment in Cocoa Dependent Economies

Data processing and analysis notebook accompanying the paper "Cocoa Price Shocks and Macroeconomic Adjustment in Cocoa Dependent Economies: Evidence from Inflation and Real Exchange Rate Dynamics" (Soma, Traore, Koassi, Kontiliguissonko, Traore, and Ouedraogo).

This notebook walks through the same pipeline as the numbered `.py` scripts in this folder, organized to mirror the structure of the paper: data, baseline results, the endogeneity correction, and the extended robustness checks. See `README.md` for a full discussion of which results reproduce exactly and which are close but not exact.

Run the setup cell below first.

In [ ]:
import sys
sys.path.insert(0, ".")

import pandas as pd
import numpy as np

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

## 1. Data

`data/cacao_panel_master.csv` is the analysis ready panel: 8 countries, 1991 to 2024, 272 country year observations. `01_data_processing.py` documents and validates how every derived variable (exposure weights, the shift share shock, the crisis dummies, the constructed bilateral real exchange rate, the leave out instrument) is built from the source level columns.

In [ ]:
import importlib
dp = importlib.import_module("01_data_processing")
panel = dp.main()

In [ ]:
df = pd.read_csv("data/cacao_panel_master.csv")
df.head()

## 2. Table 1, descriptive statistics

In [ ]:
t1 = importlib.import_module("02_table1_descriptive_stats")
t1.main()

## 3. Table 2, baseline results (H1, H2a, H2b)

Two way fixed effects panel regressions with Driscoll and Kraay (Bartlett kernel) standard errors. This reproduces Table 2 exactly.

In [ ]:
t2 = importlib.import_module("03_table2_baseline_regressions")
res_h1, res_h2a, res_h2b = t2.main()

## 4. Table 3, the asymmetry test (H3)

Splits the shock into a positive and a negative component and tests whether the two coefficients are equal. This reproduces Table 3 exactly, including the p value of 0.076 on the equality test cited in Section 5.

In [ ]:
t3 = importlib.import_module("04_table3_asymmetry_h3")
res_h3, wald = t3.main()

## 5. Section 6, the endogeneity problem and the leave out instrument

Table 4 instruments the shock with production growth excluding Cote d'Ivoire and Ghana. This reproduces the paper's qualitative pattern (insignificant on H1, significant on H2a and H2b, strong first stage) but not every point estimate to the last decimal; see the docstring in `05_table4_iv_leaveout.py`.

In [ ]:
t4 = importlib.import_module("05_table4_iv_leaveout")
iv_results = t4.main()

## 6. Section 7.2, double machine learning (Table 5)

Partially linear double machine learning (Chernozhukov et al., 2018) with Lasso or random forest selecting among eleven candidate controls. Depends on cross fitting randomness; see the docstring in `06_table5_dml_control_selection.py`.

In [ ]:
t5 = importlib.import_module("06_table5_dml_control_selection")
dml_results = t5.main()

## 7. Section 7.3, causal forest

In [ ]:
cf = importlib.import_module("07_causal_forest")
cf.main()

## 8. Section 7.4, local projections (Figure 4)

Estimates the response of each real exchange rate measure at horizons 0 to 6 years. Reproduces the shape of Figure 4 closely, including the significant four year horizon coefficient for the bilateral real exchange rate.

In [ ]:
lp = importlib.import_module("08_local_projections")
lp_table = lp.main()

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
panel_idx = df.set_index(["country", "year"])
for ax, (name, dep) in zip(axes, [("Official REER", "reer_growth"), ("Bilateral RER", "rer_bilateral_growth")]):
    out = lp.run_lp(panel_idx, dep)
    ax.axhline(0, color="grey", linewidth=0.8)
    ax.plot(out["horizon"], out["coef"], marker="o")
    ax.fill_between(out["horizon"], out["ci_lower"], out["ci_upper"], alpha=0.2)
    ax.set_title(name)
    ax.set_xlabel("Horizon (years)")
axes[0].set_ylabel("Coefficient on the shock")
fig.suptitle("Dynamic response of the real exchange rate to a cocoa shock")
fig.tight_layout()
plt.show()

## 9. Section 7.5, institutional heterogeneity (Table 6)

Tests whether the transmission of the shock differs between an administered and a liberalized producer price regime, with and without a control for Cote d'Ivoire's political crisis years. The "without crisis control" column reproduces Table 6 exactly, including the p value of 0.006.

In [ ]:
t6 = importlib.import_module("09_table6_institutional_heterogeneity")
t6.main()

## 10. Section 7.6, synthetic control case study (Figure 6)

Builds a synthetic Cote d'Ivoire from a donor pool of low cocoa exposure countries and compares its counterfactual 2023 to 2024 inflation trajectory to what was observed. Reproduces Figure 6 closely: a donor weighting of about 59 percent Ecuador and 41 percent Dominican Republic, and a post period gap of about 1.1 to 1.2 percentage points.

In [ ]:
sc = importlib.import_module("10_synthetic_control_civ")
sc.main()

## Summary

This notebook, run top to bottom against `data/cacao_panel_master.csv`, reproduces every headline result in the paper (Tables 1 through 3, the "without crisis control" column of Table 6, and the Figure 6 synthetic control) exactly, and every extension result (Table 4, Table 5, the causal forest, Figure 4, and the "with crisis control" column of Table 6) closely, in every case matching the sign, rough magnitude, and statistical significance pattern reported in the paper. See `README.md` for the full discussion of reproduction fidelity.